# Turkish chunk prep for POWSM LoRA

**Local:** set the notebook’s working directory to `sig/fine-tune` or `sig/fine-tune/notebooks` (or `cd` there in the terminal before `jupyter`), so the first cell can import `turkish_lora_util.py`.

Reads WAV + TextGrid from `sig/fine-tune/data/task-1` and `task-2`, writes 20 s / 16 kHz chunks to `sig/fine-tune/data/turkish_chunks/`.

- **Phone tiers:** task-1 → `phones`, task-2 → `REF-phones`.
- **Chunking:** phones whose interval **midpoint** lies in `[t, t+20s)` are included (avoids dropping boundary spans).
- Optional **`phone_map.json`** aliases are applied if present (run the vocab cell after first prep, edit aliases, re-run prep if needed).

In [4]:
from __future__ import annotations

import json
import sys
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "turkish_lora_util.py").is_file():
    fine_tune_root = _here
elif (_here.parent / "turkish_lora_util.py").is_file():
    fine_tune_root = _here.parent
else:
    raise SystemExit(
        "Run this notebook with cwd = sig/fine-tune or sig/fine-tune/notebooks"
    )
sys.path.insert(0, str(fine_tune_root))

from turkish_lora_util import (
    DEFAULT_CHUNKS_DIR,
    DEFAULT_RAW_DATA,
    PHONE_TIER_BY_TASK,
    chunk_corpus,
    load_phone_map,
    write_splits,
)

DATA_DIRS = {
    "task1": DEFAULT_RAW_DATA / "task-1",
    "task2": DEFAULT_RAW_DATA / "task-2",
}
OUT_DIR = DEFAULT_CHUNKS_DIR
PHONE_MAP_PATH = OUT_DIR / "phone_map.json"

print("PHONE_TIER_BY_TASK:", PHONE_TIER_BY_TASK)
for k, p in DATA_DIRS.items():
    print(k, "exists" if p.is_dir() else "MISSING", p)

SystemExit: Run this notebook with cwd = sig/fine-tune or sig/fine-tune/notebooks

In [3]:
# Optional: print TextGrid tier names (spot-check; no strict TextGrid parser)
import re


def _read_tg_unicode(p: Path) -> str:
    b = p.read_bytes()
    if b.startswith(b"\xff\xfe") or b.startswith(b"\xfe\xff"):
        return b.decode("utf-16", errors="replace")
    return b.decode("utf-8", errors="replace")


for task, d in DATA_DIRS.items():
    sample = next(iter(sorted(d.glob("*.TextGrid"))), None)
    if not sample:
        continue
    raw = _read_tg_unicode(sample)
    names = re.findall(r'name = "([^"]+)"', raw)
    print(task, sample.name, names)

NameError: name 'DATA_DIRS' is not defined

In [3]:
pmap = load_phone_map(PHONE_MAP_PATH)
manifest = chunk_corpus(DATA_DIRS, OUT_DIR, phone_map=pmap or None)

(OUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
write_splits(OUT_DIR, manifest)

train = json.loads((OUT_DIR / "train.json").read_text(encoding="utf-8"))
val = json.loads((OUT_DIR / "val.json").read_text(encoding="utf-8"))
test = json.loads((OUT_DIR / "test.json").read_text(encoding="utf-8"))
print("total chunks", len(manifest))
print("train/val/test", len(train), len(val), len(test))

total chunks 1156
train/val/test 998 75 83


In [4]:
# Verify WAV length (320_000 samples @ 16 kHz × 20 s)
import soundfile as sf

bad = []
for c in manifest[:50]:
    a, r = sf.read(OUT_DIR / f"{c['id']}.wav")
    if r != 16000 or a.shape[0] != 320_000:
        bad.append((c["id"], r, a.shape[0]))
print("sample check (first 50):", "ok" if not bad else bad)

sample check (first 50): ok


## POWSM vocabulary audit → `phone_map.json`

Requires `espnet` + `espnet_model_zoo`. Fill `aliases` so every TextGrid phone maps to a token **without** slashes (same string POWSM uses inside `/.../`).

Re-run the prep cell after editing the map if aliases change.

In [5]:
from espnet2.bin.s2t_inference import Speech2Text

s2t = Speech2Text.from_pretrained(
    "espnet/powsm",
    device="cpu",
    lang_sym="<unk>",
    task_sym="<pr>",
)
vocab = set(s2t.converter.token2id.keys())
powsm_phones = {t.strip("/") for t in vocab if t.startswith("/") and t.endswith("/")}

tg_phones = {p for c in manifest for p in c["phones"]}
unknown = sorted(tg_phones - powsm_phones)
print("Unique phones in manifest:", len(tg_phones))
print("Unknown vs POWSM slash-tokens:", len(unknown))
print(unknown[:40], "..." if len(unknown) > 40 else "")

prev: dict = {}
if PHONE_MAP_PATH.is_file():
    prev = json.loads(PHONE_MAP_PATH.read_text(encoding="utf-8"))

payload = {
    "description": "Map TextGrid phone label -> POWSM phone token (no slashes). Empty means identity.",
    "aliases": prev.get("aliases", {}),
    "unique_textgrid_phones": prev.get("unique_textgrid_phones")
    or sorted(tg_phones),
    "unknown_before_mapping": unknown,
}
PHONE_MAP_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", PHONE_MAP_PATH)

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Unique phones in manifest: 122
Unknown vs POWSM slash-tokens: 44
['ai', 'aj', 'aw', 'aʊ', 'dv', 'dʒ', 'ej', 'eı', 'g', 'his', 'i:', 'lɪ', 'nsıd', 'nt', 'nə', 'ow', 'pt', 'rd', 'retn', 'rk', 'rt', 'spn', 'st', 't ö d', 'tʃ', 'u:', 'zing', 'ç', 'ö', 'ı', 'ɑj', 'ɑw', 'ɔj', 'ə-', 'əɾ', 'ɚ', 'ɛj', 'ɜː d', 'ɝ', 'ʉ:'] ...
Wrote C:\Users\faruq\Desktop\college\senior\sig\fine-tune\data\turkish_chunks\phone_map.json
